# Loan Approval Model: Train and Save
### Model Deployment Session

This notebook trains a loan approval model and saves it so we can
deploy it as a live API. This is where deployment starts, with a saved model file.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import joblib
import shap
import matplotlib.pyplot as plt

## Step 1: Create the dataset

In [ ]:
np.random.seed(42)
n = 500

income        = np.random.randint(15000, 120000, n)
age           = np.random.randint(21, 65, n)
loan_amount   = np.random.randint(5000, 80000, n)
credit_score  = np.random.randint(300, 850, n)
existing_debt = np.random.randint(0, 50000, n)

score = (
    (credit_score / 850) * 0.45 +
    (income / 120000) * 0.30 +
    (1 - existing_debt / 50000) * 0.15 +
    (1 - loan_amount / 80000) * 0.10
)
approved = (score > 0.50).astype(int)

df = pd.DataFrame({
    'income':        income,
    'age':           age,
    'loan_amount':   loan_amount,
    'credit_score':  credit_score,
    'existing_debt': existing_debt,
    'approved':      approved
})

print("Dataset shape:", df.shape)
print(f"Approval rate: {df['approved'].mean():.1%}")
df.head()

Dataset shape: (500, 6)
Approval rate: 76.6%


,income,age,loan_amount,credit_score,existing_debt,approved
0,30795,25,52362,753,40194,1
1,15860,42,25128,336,8533,0
2,118694,49,64383,615,32581,1
3,91820,23,70545,413,5249,1
4,69886,32,14865,339,19201,1


## Step 2: Train the model

In [ ]:
X = df.drop('approved', axis=1)
y = df['approved']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

acc = accuracy_score(y_test, model.predict(X_test))
print(f"Model accuracy on test set: {acc:.1%}")

Model accuracy on test set: 92.0%


## Step 3: Save the model

This is the most important step for deployment.
The .pkl file is what the API will load to make predictions.

In [ ]:
joblib.dump(model, 'loan_model.pkl')
print("Model saved as loan_model.pkl")

Model saved as loan_model.pkl


## Step 4: Test a prediction before deploying

Always verify the model works locally before building the API.
If it does not work here it will not work in deployment either.

In [ ]:
g

## Step 5: Explain the prediction with SHAP

SHAP tells us WHY the model made this decision, not just approved or denied,
but which features drove the outcome and by how much.

In financial services this is called model explainability.
In many countries banks are legally required to explain credit decisions.
A model that cannot explain itself cannot be deployed responsibly.

In [ ]:
explainer   = shap.TreeExplainer(model)
shap_values = explainer.shap_values(applicant)

# shap_values has shape (samples, features, classes)
# Index [:, :, 1] gives values for the Approved class
fig, ax = plt.subplots(figsize=(8, 3))
shap.waterfall_plot(
    shap.Explanation(
        values        = shap_values[0, :, 1],
        base_values   = explainer.expected_value[1],
        data          = applicant.values[0],
        feature_names = list(applicant.columns)
    ),
    show=False
)
plt.title("Why was this applicant APPROVED?", fontsize=13, pad=10)
plt.tight_layout()
plt.show()

Bars to the right push toward APPROVED.
Bars to the left push toward DENIED.
The longer the bar, the more influence that feature had.
Credit score and income drove this approval.

In [ ]:
# Which features matter most overall — across all test applicants
shap_test = explainer.shap_values(X_test)

fig, ax = plt.subplots(figsize=(7, 4))
shap.summary_plot(
    shap_test[:, :, 1],
    X_test,
    plot_type="bar",
    show=False
)
plt.title("Feature importance across all applicants", fontsize=13)
plt.tight_layout()
plt.show()

## What you now have

A trained Random Forest model saved as loan_model.pkl
A verified prediction working locally
SHAP explainability showing why each decision was made

Next step: take this loan_model.pkl and build a FastAPI backend
that loads it and serves predictions through an endpoint.